# Control Variables - 14 Night Light
18/05/2026, Kuba Kowalski 

In [3]:
# packages
from pathlib import Path
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
import numpy as np
import pandas as pd
from shapely.geometry import mapping
# from rasterstats import zonal_stats
from rasterio.mask import mask

## Tropical Africa 

### Setup

In [4]:
# -----------------------------
# Paths
# -----------------------------

base = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
)

# Input folder
input_dir = base / r"1_inputs\ancilliary_data\14_night_lights"

# Output folder
out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights"
)

out_dir.mkdir(exist_ok=True)

# Administrative boundaries
admin_path = (
    base
    / r"3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
)

# Gas flare points
flare_path = input_dir / "VIIRS_Global_flaring_d.7_slope_0.0298_2012-2016_web.kml"

# Nighttime lights rasters
ntl_paths = {
    2000: input_dir / "F152000.v4b_web.stable_lights.avg_vis.tif",
    2012: input_dir / "F182012.v4c_web.stable_lights.avg_vis.tif",
}

In [5]:
# Fiona started throwing up an error that it doesn't recognize KML. Roundabout way of fixing it 
import fiona

fiona.supported_drivers["KML"] = "rw"
fiona.supported_drivers["LIBKML"] = "rw"

admin = gpd.read_file(admin_path)

try:
    flares = gpd.read_file(flare_path, driver="KML")
except Exception:
    flares = gpd.read_file(flare_path, engine="pyogrio")

admin = admin.to_crs("EPSG:4326")
flares = flares.to_crs("EPSG:4326")

admin_union = admin.geometry.union_all()

zone_id = "GEOLEVEL1"

### Flare removal buffers

In [6]:
# -----------------------------
# Define 5-pixel buffer
# -----------------------------

cell_size_deg = 0.0083333333
buffer_pixels = 5
buffer_distance_deg = cell_size_deg * buffer_pixels

flare_buffers = flares.copy()
flare_buffers["geometry"] = flare_buffers.geometry.buffer(buffer_distance_deg)

# Dissolve into one mask geometry
flare_mask_geom = flare_buffers.dissolve()

C:\Users\kowal010\AppData\Local\Temp\ipykernel_19468\3338436297.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  flare_buffers["geometry"] = flare_buffers.geometry.buffer(buffer_distance_deg)


In [ ]:

# --------------------------------------------------
# CLEAN ADMIN GEOMETRIES
# --------------------------------------------------

admin = admin.copy()

admin["geometry"] = admin.geometry.make_valid()

admin = admin[
    admin.geometry.notna() &
    ~admin.geometry.is_empty
].copy()

print("Admin polygons after geometry cleaning:", len(admin))

all_stats = []

for year, raster_path in ntl_paths.items():

    print(f"\nProcessing {year}...")

    # --------------------------------------------------
    # OPEN AND CLIP RASTER
    # --------------------------------------------------

    with rasterio.open(raster_path) as src:

        clipped, clipped_transform = mask(
            src,
            [mapping(admin.geometry.union_all())],
            crop=True,
            nodata=255
        )

        meta = src.meta.copy()

        meta.update({
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": clipped_transform,
            "nodata": np.nan,
            "dtype": "float32"
        })

        arr = clipped[0].astype("float32")

    # --------------------------------------------------
    # RASTERIZE FLARE BUFFER
    # --------------------------------------------------

    flare_raster = rasterize(
        [(geom, 1) for geom in flare_mask_geom.geometry],
        out_shape=arr.shape,
        transform=clipped_transform,
        fill=0,
        dtype="uint8"
    )

    # --------------------------------------------------
    # MASK:
    # - flare pixels
    # - zeros
    # - 255 no-observation
    # --------------------------------------------------

    arr[
        (flare_raster == 1) |
        (arr == 0) |
        (arr == 255)
    ] = np.nan

    # --------------------------------------------------
    # SAVE CLEAN RASTER
    # --------------------------------------------------

    clean_raster_path = (
        out_dir / f"ntl_{year}_clean_no_flares.tif"
    )

    with rasterio.open(clean_raster_path, "w", **meta) as dst:
        dst.write(arr, 1)

    print(f"Saved cleaned raster: {clean_raster_path}")

    # --------------------------------------------------
    # ZONAL STATISTICS
    # --------------------------------------------------

    stats_rows = []

    with rasterio.open(clean_raster_path) as src:

        for idx, row in admin.iterrows():

            geom = [mapping(row.geometry)]

            try:

                out_image, _ = mask(
                    src,
                    geom,
                    crop=True,
                    nodata=np.nan
                )

                values = out_image[0]
                values = values[~np.isnan(values)]

                if len(values) == 0:

                    stats_rows.append({
                        zone_id: row[zone_id],
                        "year": year,
                        "count": 0,
                        "min": np.nan,
                        "max": np.nan,
                        "mean": np.nan,
                        "std": np.nan,
                        "sum": np.nan,
                        "median": np.nan,
                        "range": np.nan
                    })

                else:

                    stats_rows.append({
                        zone_id: row[zone_id],
                        "year": year,
                        "count": int(len(values)),
                        "min": float(np.min(values)),
                        "max": float(np.max(values)),
                        "mean": float(np.mean(values)),
                        "std": float(np.std(values)),
                        "sum": float(np.sum(values)),
                        "median": float(np.median(values)),
                        "range": float(np.max(values) - np.min(values))
                    })

            except Exception:

                stats_rows.append({
                    zone_id: row[zone_id],
                    "year": year,
                    "count": 0,
                    "min": np.nan,
                    "max": np.nan,
                    "mean": np.nan,
                    "std": np.nan,
                    "sum": np.nan,
                    "median": np.nan,
                    "range": np.nan
                })

    # --------------------------------------------------
    # SAVE TABLE
    # --------------------------------------------------

    stats_df = pd.DataFrame(stats_rows).rename(
        columns={"mean": f"mean_{year}"}
    )

    table_path = out_dir / f"zonal_stats_{year}.csv"

    stats_df.to_csv(table_path, index=False)

    print(f"Saved zonal stats table: {table_path}")

    # --------------------------------------------------
    # SAVE SPATIAL OUTPUT
    # --------------------------------------------------

    admin_stats = admin.merge(
        stats_df,
        on=zone_id
    )

    vector_path = out_dir / f"admin_ntl_stats_{year}.gpkg"

    admin_stats.to_file(
        vector_path,
        driver="GPKG"
    )

    print(f"Saved spatial output: {vector_path}")

    # --------------------------------------------------
    # STORE FOR PANEL
    # --------------------------------------------------

    all_stats.append(stats_df)

# ============================================================
# CREATE PANEL DATASET
# ============================================================

panel = pd.concat(all_stats, ignore_index=True)

panel_path = out_dir / "ntl_zonal_stats_panel.csv"

panel.to_csv(panel_path, index=False)

print("\nFinished all processing.")
print(f"Saved panel dataset: {panel_path}")

panel.head()

Admin polygons after geometry cleaning: 2118

Processing 2000...
Saved cleaned raster: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\ntl_2000_clean_no_flares.tif
Saved zonal stats table: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2000.csv
Saved spatial output: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\admin_ntl_stats_2000.gpkg

Processing 2012...
Saved cleaned raster: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\ntl_2012_clean_no_flares.tif
Saved zonal stats table: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_night_lights\zonal_stats_2012.csv
Saved spatial output: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\14_nigh

,GEOLEVEL1,year,count,min,max,mean,std,sum,median,range
0,204001,2000,132,5.0,15.0,6.962121,2.261018,919.0,6.0,10.0
1,204002,2000,81,5.0,15.0,8.074074,2.522872,654.0,7.0,10.0
2,204003,2000,527,4.0,38.0,8.595825,6.496665,4530.0,6.0,34.0
3,204004,2000,223,5.0,36.0,10.251122,7.056447,2286.0,7.0,31.0
4,204005,2000,149,5.0,8.0,5.765100,0.797557,859.0,6.0,3.0
